# Section 2: Network intrusion detection

Compare a Random Forest with a 1D CNN on NSL-KDD. Both models predict normal, DoS, probe, R2L or U2R.

This revision adds predictor-overlap checks, input-feature exploration, validation-loss tracking, standalone preprocessing artifacts and run provenance. Outputs were cleared; rerun from a fresh kernel, top to bottom.

## Setup

The working directory changes only in Colab. Local runs locate the repository root automatically. The run summary records package versions and dataset hashes.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    os.chdir("/content")

In [ ]:
%pip install -q joblib matplotlib numpy pandas scikit-learn torch

In [ ]:
import hashlib
import importlib.metadata
import json
import platform
import random
import shutil
import sys
import urllib.request
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, VarianceThreshold, f_classif
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, average_precision_score,
    classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

### Experiment settings

Keep the seed, epoch count and output folders together. Colab writes to its remote workspace, not directly to the local repository. Seeds help repeatability, but different environments can still produce different results.

In [ ]:
seed = 42
epochs = 8
run_id = datetime.now(timezone.utc).strftime("s02-%Y%m%dT%H%M%SZ")

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "notebooks").is_dir() and (candidate / "docs").is_dir():
            return candidate
    return Path.cwd()

in_colab = "google.colab" in sys.modules
root = Path("/content/section_02_workspace") if in_colab else find_project_root()
data_dir = root / "data/raw/nsl-kdd"
processed_dir = root / "data/processed/section_02"
model_dir = root / "models/section_02"
results_dir = root / "reports/section_02"
for directory in (data_dir, processed_dir, model_dir, results_dir / "metrics", results_dir / "figures"):
    directory.mkdir(parents=True, exist_ok=True)

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

## Load NSL-KDD

Use the official training and test partitions from the configured mirror. Existing files are reused. This download step does not check the hashes listed in the dataset notes.

In [ ]:
base_url = "https://raw.githubusercontent.com/HoaNP/NSL-KDD-DataSet/master"
for filename in ("KDDTrain+.txt", "KDDTest+.txt"):
    path = data_dir / filename
    if not path.exists():
        urllib.request.urlretrieve(f"{base_url}/{filename.replace('+', '%2B')}", path)
dataset_hashes = {
    filename: hashlib.sha256((data_dir / filename).read_bytes()).hexdigest()
    for filename in ("KDDTrain+.txt", "KDDTest+.txt")
}

In [ ]:
# The text files have 41 input features, followed by the attack name and difficulty.
feature_cols = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login",
    "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
    "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate",
]
cat_cols = ["protocol_type", "service", "flag"]
num_cols = [column for column in feature_cols if column not in cat_cols]
data_cols = [*feature_cols, "attack_name", "difficulty"]
# Group the individual attack names into the five classes used for evaluation.
class_names = ["normal", "dos", "probe", "r2l", "u2r"]
class_ids = {name: index for index, name in enumerate(class_names)}
attack_groups = {
    **{name: "dos" for name in "back land neptune pod smurf teardrop mailbomb apache2 processtable udpstorm".split()},
    **{name: "probe" for name in "ipsweep nmap portsweep satan saint mscan".split()},
    **{name: "r2l" for name in "ftp_write guess_passwd imap multihop phf spy warezclient warezmaster sendmail named snmpgetattack snmpguess xlock xsnoop worm".split()},
    **{name: "u2r" for name in "buffer_overflow loadmodule perl rootkit ps sqlattack xterm httptunnel".split()},
    "normal": "normal",
}

def load_data(filename):
    frame = pd.read_csv(data_dir / filename, names=data_cols)
    # Clean up label spelling before applying the class mapping.
    frame["attack_name"] = frame["attack_name"].str.strip().str.lower().str.rstrip(".")
    frame["label_name"] = frame["attack_name"].map(attack_groups)
    frame["label"] = frame["label_name"].map(class_ids).astype(int)
    return frame

### Split and inspect the data

Keep the official test set untouched and reserve 15% of official training data for validation. Profile missingness, full-record duplicates, predictor-only duplicates, cross-partition predictor overlap, skewness and representative feature distributions before fitting.

In [ ]:
full_train = load_data("KDDTrain+.txt")
test = load_data("KDDTest+.txt")
train, val = train_test_split(
    full_train, test_size=0.15, stratify=full_train["label"], random_state=seed,
)
train, val = train.reset_index(drop=True), val.reset_index(drop=True)

profile = pd.DataFrame([
    {"split": name, "records": len(frame), "missing": int(frame.isna().sum().sum()),
     "full_record_duplicates": int(frame[data_cols].duplicated().sum()),
     "predictor_duplicates": int(frame[feature_cols].duplicated().sum()),
     **frame["label_name"].value_counts().reindex(class_names, fill_value=0).to_dict()}
    for name, frame in (("train", train), ("validation", val), ("test", test))
])

predictor_hashes = {
    name: set(pd.util.hash_pandas_object(frame[feature_cols], index=False).astype(str))
    for name, frame in (("train", train), ("validation", val), ("test", test))
}
overlap = {
    "train_validation_predictor_overlap": len(predictor_hashes["train"] & predictor_hashes["validation"]),
    "train_test_predictor_overlap": len(predictor_hashes["train"] & predictor_hashes["test"]),
    "validation_test_predictor_overlap": len(predictor_hashes["validation"] & predictor_hashes["test"]),
}
profile.to_json(processed_dir / "data-profile.json", orient="records", indent=2)
(processed_dir / "predictor-overlap.json").write_text(json.dumps(overlap, indent=2))
display(profile)
display(pd.Series(overlap).to_frame("matching predictor rows"))

skewness = train[num_cols].skew(numeric_only=True).sort_values(key=np.abs, ascending=False)
skewness.rename("skewness").to_csv(processed_dir / "numeric-skewness.csv")
correlations = train[num_cols].corr(numeric_only=True)
correlation_pairs = correlations.where(np.triu(np.ones(correlations.shape), 1).astype(bool)).stack()
correlation_pairs.reindex(correlation_pairs.abs().sort_values(ascending=False).index).head(50).rename(
    "correlation"
).to_csv(processed_dir / "top-numeric-correlations.csv")
plot_cols = [column for column in ["duration", "src_bytes", "dst_bytes", "count"] if column in train]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for column, axis in zip(plot_cols, axes.flat):
    axis.hist(np.log1p(train[column].clip(lower=0)), bins=50)
    axis.set_title(f"log1p({column})")
fig.tight_layout()
fig.savefig(results_dir / "figures/input-feature-distributions.png", dpi=180)
plt.show()

## Prepare the features

Standardize the numerical columns, one-hot encode the categories, remove constant columns and select 64 features using ANOVA. Fit these steps on training data only, then reuse them for validation and testing.

The saved profile has no missing values, so this notebook does not impute them. The duplicate check covers complete records within each split; it does not check matching predictor vectors across splits.

In [ ]:
# Scale numeric features and turn each categorical value into a separate indicator.
column_prep = ColumnTransformer([
    ("numeric", StandardScaler(), num_cols),
    ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
], verbose_feature_names_out=False)
prep = Pipeline([
    ("columns", column_prep),
    # Drop features that have no variation in the training data.
    ("variance", VarianceThreshold()),
    # Keep the 64 features with the strongest individual class associations.
    ("selection", SelectKBest(f_classif, k=64)),
])

# Learn the preprocessing from training rows, then apply it unchanged to the other splits.
X_train = prep.fit_transform(train[feature_cols], train["label"]).astype(np.float32)
X_val = prep.transform(val[feature_cols]).astype(np.float32)
X_test = prep.transform(test[feature_cols]).astype(np.float32)
y_train = train["label"].to_numpy(np.int64)
y_val = val["label"].to_numpy(np.int64)
y_test = test["label"].to_numpy(np.int64)

# Follow the selection masks to record which feature names survived.
feature_names = prep.named_steps["columns"].get_feature_names_out()
feature_names = feature_names[prep.named_steps["variance"].get_support()]
feature_names = feature_names[prep.named_steps["selection"].get_support()].tolist()
(processed_dir / "selected-features.json").write_text(json.dumps(feature_names, indent=2))
display(pd.DataFrame({"split": ["train", "validation", "test"],
                      "records": [len(X_train), len(X_val), len(X_test)],
                      "features": [X_train.shape[1]] * 3}))

## Evaluate the models

Choose the class with the highest score. Macro averages give each class equal weight; weighted F1 reflects class size. The confusion matrix and one-versus-rest precision-recall curves show which attacks are being missed.

In [ ]:
def show_results(name, labels, probs, filename):
    # Pick the highest-scoring class for each connection.
    preds = probs.argmax(axis=1)
    # Treat each class as positive in turn when drawing its precision-recall curve.
    binary_labels = label_binarize(labels, classes=np.arange(len(class_names)))
    # Save overall and per-class scores so rare attacks are not hidden by accuracy.
    metrics = {
        "run_id": run_id,
        "accuracy": accuracy_score(labels, preds),
        "macro_precision": precision_score(labels, preds, average="macro", zero_division=0),
        "macro_recall": recall_score(labels, preds, average="macro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "weighted_f1": f1_score(labels, preds, average="weighted", zero_division=0),
        "macro_average_precision": average_precision_score(binary_labels, probs, average="macro"),
        "confusion_matrix": confusion_matrix(labels, preds).tolist(),
        "classification_report": classification_report(
            labels, preds, labels=np.arange(len(class_names)),
            target_names=class_names, output_dict=True, zero_division=0,
        ),
    }
    (results_dir / "metrics" / f"{filename}.json").write_text(json.dumps(metrics, indent=2))

    # Show the classification errors next to the precision-recall curves.
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay.from_predictions(
        labels, preds, display_labels=class_names, cmap="Blues", colorbar=False, ax=axes[0],
    )
    axes[0].set_title("Confusion matrix")
    for index, class_name in enumerate(class_names):
        precision, recall, _ = precision_recall_curve(binary_labels[:, index], probs[:, index])
        axes[1].plot(recall, precision, label=class_name)
    axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-recall curves")
    axes[1].legend()
    fig.suptitle(name)
    fig.tight_layout()
    fig.savefig(results_dir / "figures" / f"{filename}-evaluation.png", dpi=180)
    plt.show()
    return metrics

## Random Forest

Train 180 trees with balanced subsample weights to give rarer classes more influence. Both models use the same selected features; the labels and difficulty column are not predictors.

In [ ]:
# Build a forest with class balancing inside each bootstrap sample.
rf_model = RandomForestClassifier(
    n_estimators=180, max_depth=28, max_features="sqrt",
    class_weight="balanced_subsample", n_jobs=1, random_state=seed,
)
# Train on the selected features and score the held-out test connections.
rf_model.fit(X_train, y_train)
rf_probs = rf_model.predict_proba(X_test)
rf_metrics = show_results("Random Forest", y_test, rf_probs, "random-forest")
# Save the fitted preprocessing with the forest for use on new connections.
joblib.dump({"feature_pipeline": prep, "model": rf_model, "class_names": class_names, "run_id": run_id},
            model_dir / "random-forest.joblib")
display(pd.DataFrame(rf_metrics["classification_report"]).T)

## CNN

Use two convolutional layers followed by pooling, dropout and a five-class output. Here the convolution moves across feature columns, not a sequence of packets. Neighbouring columns do not necessarily have a natural spatial relationship.

In [ ]:
class FeatureCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # Learn local patterns across the feature columns, then pool them into five class scores.
        self.network = nn.Sequential(
            nn.Conv1d(1, 32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(),
            nn.Conv1d(32, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.AdaptiveMaxPool1d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, 5),
        )

    def forward(self, values):
        # Add the channel dimension expected by Conv1d: batch, channel, features.
        return self.network(values.unsqueeze(1))

def make_loader(features, labels, shuffle=False):
    # Pair each feature row with its class label for batching.
    dataset = TensorDataset(torch.from_numpy(features), torch.from_numpy(labels))
    return DataLoader(dataset, batch_size=512, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed) if shuffle else None)

### Set up training

Prepare batches of the selected features and use CUDA if available. Class weights come from training counts, with a square root to soften the effect of very rare classes. Cross-entropy uses logits; softmax is only needed for prediction scores.

In [ ]:
# Shuffle training batches while keeping validation and test order fixed.
train_loader = make_loader(X_train, y_train, True)
val_loader = make_loader(X_val, y_val)
test_loader = make_loader(X_test, y_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cnn = FeatureCNN().to(device)
# Give rare classes more weight, using training counts only.
class_counts = np.bincount(y_train, minlength=5)
class_weights = np.sqrt(len(y_train) / (5 * class_counts))
class_weights = class_weights / class_weights.mean()
loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32, device=device))
optimizer = torch.optim.Adam(cnn.parameters(), lr=0.001, weight_decay=0.0001)

def predict_cnn(loader):
    # Turn off dropout and freeze batch-normalization statistics during prediction.
    cnn.eval()
    probs = []
    with torch.no_grad():
        for values, _ in loader:
            probs.append(torch.softmax(cnn(values.to(device)), dim=1).cpu().numpy())
    return np.concatenate(probs)

### Train and evaluate

Run all configured epochs, record training loss, validation loss and validation macro F1, then restore the checkpoint with the best validation macro F1. Save the numerical history and chosen epoch. The preprocessing pipeline is saved independently with the CNN weights.

In [ ]:
def validation_loss(loader):
    cnn.eval()
    total = 0.0
    with torch.no_grad():
        for values, labels in loader:
            values, labels = values.to(device), labels.to(device)
            total += loss_fn(cnn(values), labels).item() * len(labels)
    return total / len(loader.dataset)

history = []
best_val_f1 = -1
for epoch in range(1, epochs + 1):
    cnn.train()
    total_loss = 0
    for values, labels in train_loader:
        values, labels = values.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = loss_fn(cnn(values), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)

    val_probs = predict_cnn(val_loader)
    val_f1 = f1_score(y_val, val_probs.argmax(1), average="macro")
    val_loss = validation_loss(val_loader)
    history.append({"epoch": epoch, "training_loss": total_loss / len(train),
                    "validation_loss": val_loss, "validation_macro_f1": val_f1})
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        chosen_epoch = epoch
        best_state = {name: value.detach().cpu().clone() for name, value in cnn.state_dict().items()}

cnn.load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.to_csv(results_dir / "metrics/cnn-training-history.csv", index=False)
display(history_df)
history_df.set_index("epoch").plot(subplots=True, figsize=(7, 7))
plt.tight_layout()
plt.savefig(results_dir / "figures/cnn-training-history.png", dpi=180)
plt.show()

cnn_probs = predict_cnn(test_loader)
cnn_metrics = show_results("1D CNN", y_test, cnn_probs, "cnn")
cnn_metrics.update({"run_id": run_id, "best_validation_macro_f1": best_val_f1,
                    "chosen_epoch": chosen_epoch, "epochs": epochs, "device": str(device)})
(results_dir / "metrics/cnn.json").write_text(json.dumps(cnn_metrics, indent=2))
joblib.dump(prep, model_dir / "feature-preprocessor.joblib")
torch.save({"model_state": best_state, "class_names": class_names,
            "feature_names": feature_names, "run_id": run_id}, model_dir / "cnn.pt")
display(pd.DataFrame(cnn_metrics["classification_report"]).T)

## Compare the models

Save test metrics, numerical histories, profiles, selected features, fitted preprocessing, models and environment metadata under one run identifier. The CNN still convolves over selected columns rather than packets; its locality assumption remains a model limitation to discuss.

In [ ]:
metric_names = ["accuracy", "macro_precision", "macro_recall", "macro_f1", "weighted_f1", "macro_average_precision"]
comparison = pd.DataFrame([
    {"model": "Random Forest", **{metric: rf_metrics[metric] for metric in metric_names}},
    {"model": "1D CNN", **{metric: cnn_metrics[metric] for metric in metric_names}},
])
comparison.to_csv(results_dir / "model-comparison.csv", index=False)
packages = ["joblib", "matplotlib", "numpy", "pandas", "scikit-learn", "torch"]
(results_dir / "run-summary.json").write_text(json.dumps({
    "run_id": run_id, "created_utc": datetime.now(timezone.utc).isoformat(), "seed": seed,
    "dataset_sha256": dataset_hashes,
    "train_records": len(train), "validation_records": len(val), "test_records": len(test),
    "selected_features": X_train.shape[1], "cnn_epochs": epochs, "cnn_chosen_epoch": chosen_epoch,
    "python": platform.python_version(), "platform": platform.platform(),
    "packages": {name: importlib.metadata.version(name) for name in packages},
}, indent=2))
display(comparison.style.format({metric: "{:.4f}" for metric in metric_names}).highlight_max(subset=metric_names, color="#d9ead3"))

if in_colab:
    export_dir = root / "section_02_export"
    shutil.copytree(results_dir, export_dir / "reports", dirs_exist_ok=True)
    shutil.copytree(model_dir, export_dir / "models", dirs_exist_ok=True)
    shutil.copytree(processed_dir, export_dir / "processed", dirs_exist_ok=True)
    shutil.make_archive("/content/section_02_results", "zip", root_dir=export_dir)

## What to take from the results

- Look at macro F1 and per-class recall alongside accuracy. R2L and U2R are rare, and a good overall score can hide missed attacks.
- The official test set includes attack types absent from training. NSL-KDD is also an old benchmark, so these results do not establish performance on current traffic.
- Stored notebook outputs and saved report files can come from different runs. Keep a single run's evidence together when writing up results.

See the [alignment review](../../docs/assignment/LECTURE_NOTEBOOK_ALIGNMENT.md) for the remaining methodological issues. This style revision does not change the experiment.